# 02 - Exploratory Data Analysis: Drivers & Customers

This notebook explores driver and customer data to identify patterns and potential fraud actors.

## Objectives
- Analyze driver demographics and delivery performance
- Analyze customer demographics and order behavior
- Identify high-risk drivers and customers
- Detect potential collusion patterns between drivers and customers

## Data Source
- PostgreSQL database: `walmart-delivery-fraud-detection`
- Schema: `walmart_fraud`
- Period: January 2023 - December 2023
- Region: Central Florida

In [1]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import warnings

import sys
sys.path.insert(0, '..')

from src.database.connection import (
    load_orders, load_drivers, load_customers, test_connection
)
from src.features.driver_features import (
    create_driver_features, get_suspicious_drivers, get_driver_risk_scores
)
from src.features.customer_features import (
    create_customer_features, get_suspicious_customers, get_customer_risk_scores
)
from src.features.aggregations import create_driver_customer_matrix

# Import visualization theme
from src.config.viz_theme import (
    PROJECT_THEME, apply_project_theme
)

# Settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

# Plotly default template
import plotly.io as pio
pio.templates.default = 'plotly_white'

print('Libraries loaded successfully!')

Libraries loaded successfully!


## 1. Database Connection and Data Loading

In [2]:
# Test database connection
if test_connection():
    print('Database connection: OK')
else:
    raise Exception('Database connection failed!')

Database connection: OK


In [3]:
# Load data from PostgreSQL
orders = load_orders()
drivers = load_drivers()
customers = load_customers()

print('Data loaded from PostgreSQL:')
print(f'  Orders: {len(orders):,} records')
print(f'  Drivers: {len(drivers):,} records')
print(f'  Customers: {len(customers):,} records')

Data loaded from PostgreSQL:
  Orders: 10,000 records
  Drivers: 1,247 records
  Customers: 1,239 records


In [4]:
# Preview data
print('DRIVERS SAMPLE:')
display(drivers.head())

print('\nCUSTOMERS SAMPLE:')
display(customers.head())

DRIVERS SAMPLE:


,driver_id,driver_name,age,trips
0,WDID09872,Tonya Chung,18,35
1,WDID09873,Pamela Moore,18,64
2,WDID09874,Billy Lawson,18,37
3,WDID09875,Stephen Randolph,18,64
4,WDID09876,Jordan Daniel,18,53



CUSTOMERS SAMPLE:


,customer_id,customer_name,customer_age
0,WCID5000,Dustin Spencer,38
1,WCID5001,Clayton Huff,66
2,WCID5002,Angelica Wilson,72
3,WCID5003,Lisa Marquez,20
4,WCID5004,James Martinez,35


---
## 2. Driver Analysis

Analyzing driver demographics and delivery performance to identify potential fraud patterns.

In [5]:
# Create driver features
driver_features = create_driver_features(drivers, orders)

print(f'Driver features created: {len(driver_features)} drivers')
print(f'\nFeatures: {list(driver_features.columns)}')
driver_features.head()

Driver features created: 1247 drivers

Features: ['driver_id', 'driver_name', 'age', 'trips', 'total_orders', 'total_items_delivered', 'total_items_missing', 'total_revenue', 'avg_order_value', 'missing_rate', 'orders_with_missing', 'pct_orders_with_missing', 'age_group', 'experience_level']


,driver_id,driver_name,age,trips,total_orders,total_items_delivered,total_items_missing,total_revenue,avg_order_value,missing_rate,orders_with_missing,pct_orders_with_missing,age_group,experience_level
0,WDID09872,Tonya Chung,18,35,7,104,0,2038.81,291.26,0.00,0,0.00,18-25,Intermediate
1,WDID09873,Pamela Moore,18,64,11,94,6,3858.95,350.81,6.00,4,36.36,18-25,Experienced
2,WDID09874,Billy Lawson,18,37,11,103,6,2858.31,259.85,5.50,4,36.36,18-25,Intermediate
3,WDID09875,Stephen Randolph,18,64,11,86,6,2832.49,257.50,6.52,4,36.36,18-25,Experienced
4,WDID09876,Jordan Daniel,18,53,11,99,6,2935.32,266.85,5.71,4,36.36,18-25,Experienced


In [6]:
# Driver Overview Statistics
print('=' * 60)
print('DRIVER OVERVIEW')
print('=' * 60)
print(f"Total Drivers: {len(driver_features):,}")
print(f"Active Drivers (with orders): {(driver_features['total_orders'] > 0).sum():,}")
print(f"Average Age: {driver_features['age'].mean():.1f} years")
print(f"Average Trips: {driver_features['trips'].mean():.1f}")
print(f"\nDelivery Stats:")
print(f"  Average Orders per Driver: {driver_features['total_orders'].mean():.1f}")
print(f"  Average Missing Rate: {driver_features['missing_rate'].mean():.2f}%")
print(f"  Drivers with Missing Issues: {(driver_features['orders_with_missing'] > 0).sum():,}")

DRIVER OVERVIEW
Total Drivers: 1,247
Active Drivers (with orders): 1,247
Average Age: 34.2 years
Average Trips: 45.7

Delivery Stats:
  Average Orders per Driver: 8.0
  Average Missing Rate: 1.22%
  Drivers with Missing Issues: 425


In [7]:
# Driver Age Distribution
fig = px.histogram(
    driver_features, 
    x='age',
    nbins=25,
    title='Driver Age Distribution',
    labels={'age': 'Age (years)', 'count': 'Number of Drivers'},
    color_discrete_sequence=[PROJECT_THEME['walmart_blue']]
)
fig.add_vline(x=driver_features['age'].mean(), line_dash='dash', line_color='red',
              annotation_text=f"Mean: {driver_features['age'].mean():.1f}")
fig.update_layout(height=400)
apply_project_theme(fig)
fig.show()

In [8]:
# Driver Experience Distribution
fig = px.histogram(
    driver_features, 
    x='trips',
    nbins=30,
    title='Driver Experience (Total Trips in 2023)',
    labels={'trips': 'Number of Trips', 'count': 'Number of Drivers'},
    color_discrete_sequence=[PROJECT_THEME['walmart_blue']]
)
fig.add_vline(x=driver_features['trips'].mean(), line_dash='dash', line_color='red',
              annotation_text=f"Mean: {driver_features['trips'].mean():.1f}")
fig.update_layout(height=400)
apply_project_theme(fig)
fig.show()

In [9]:
# Driver Missing Rate Distribution
active_drivers = driver_features[driver_features['total_orders'] > 0]

fig = px.histogram(
    active_drivers, 
    x='missing_rate',
    nbins=30,
    title='Distribution of Missing Rate by Driver (Active Drivers Only)',
    labels={'missing_rate': 'Missing Rate (%)', 'count': 'Number of Drivers'},
    color_discrete_sequence=[PROJECT_THEME['walmart_blue']]
)
avg_rate = active_drivers['missing_rate'].mean()
fig.add_vline(x=avg_rate, line_dash='dash', line_color='red',
              annotation_text=f"Mean: {avg_rate:.2f}%")
fig.update_layout(height=400)
apply_project_theme(fig)
fig.show()

print(f"\nDrivers with missing rate > 20%: {(active_drivers['missing_rate'] > 20).sum()}")
print(f"Drivers with missing rate > 30%: {(active_drivers['missing_rate'] > 30).sum()}")
print(f"Drivers with missing rate > 50%: {(active_drivers['missing_rate'] > 50).sum()}")


Drivers with missing rate > 20%: 0
Drivers with missing rate > 30%: 0
Drivers with missing rate > 50%: 0


In [10]:
# Missing Rate vs Experience
fig = px.scatter(
    active_drivers,
    x='trips',
    y='missing_rate',
    color='age_group',
    size='total_orders',
    hover_data=['driver_name', 'total_items_missing', 'orders_with_missing'],
    title='Missing Rate vs Driver Experience',
    labels={'trips': 'Total Trips', 'missing_rate': 'Missing Rate (%)', 'age_group': 'Age Group'}
)
fig.update_layout(height=500)
apply_project_theme(fig)
fig.show()

In [11]:
# Missing Rate by Age Group
age_analysis = active_drivers.groupby('age_group', observed=True).agg({
    'missing_rate': 'mean',
    'driver_id': 'count',
    'total_orders': 'sum',
    'total_items_missing': 'sum'
}).reset_index()
age_analysis.columns = ['age_group', 'avg_missing_rate', 'driver_count', 'total_orders', 'total_missing']

fig = px.bar(
    age_analysis,
    x='age_group',
    y='avg_missing_rate',
    title='Average Missing Rate by Driver Age Group',
    labels={'avg_missing_rate': 'Average Missing Rate (%)', 'age_group': 'Age Group'},
    color='avg_missing_rate',
    color_continuous_scale='Reds',
    text='avg_missing_rate'
)
fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(height=400)
apply_project_theme(fig)
fig.show()

print('\nAge Group Details:')
display(age_analysis)


Age Group Details:


,age_group,avg_missing_rate,driver_count,total_orders,total_missing
0,18-25,1.07,549,4351,647
1,26-35,1.35,186,1507,275
2,36-45,1.24,163,1304,221
3,46-55,1.30,177,1432,254
4,55+,1.42,172,1406,260


In [12]:
# Missing Rate by Experience Level
exp_analysis = active_drivers.groupby('experience_level', observed=True).agg({
    'missing_rate': 'mean',
    'driver_id': 'count',
    'total_orders': 'sum',
    'total_items_missing': 'sum'
}).reset_index()
exp_analysis.columns = ['experience_level', 'avg_missing_rate', 'driver_count', 'total_orders', 'total_missing']

fig = px.bar(
    exp_analysis,
    x='experience_level',
    y='avg_missing_rate',
    title='Average Missing Rate by Experience Level',
    labels={'avg_missing_rate': 'Average Missing Rate (%)', 'experience_level': 'Experience Level'},
    color='avg_missing_rate',
    color_continuous_scale='Oranges',
    text='avg_missing_rate'
)
fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(height=400)
apply_project_theme(fig)
fig.show()

In [13]:
# Identify Suspicious Drivers
suspicious_drivers = get_suspicious_drivers(driver_features, missing_rate_threshold=15, min_orders=5)

print('=' * 60)
print('SUSPICIOUS DRIVERS')
print('=' * 60)
print(f"Total suspicious drivers: {len(suspicious_drivers)}")
print(f"Percentage of active drivers: {len(suspicious_drivers)/len(active_drivers)*100:.1f}%")
print()

# Show top 15 suspicious drivers
display(suspicious_drivers[[
    'driver_id', 'driver_name', 'age', 'trips',
    'total_orders', 'total_items_missing', 'missing_rate', 'pct_orders_with_missing'
]].head(15))

SUSPICIOUS DRIVERS
Total suspicious drivers: 54
Percentage of active drivers: 4.3%



,driver_id,driver_name,age,trips,total_orders,total_items_missing,missing_rate,pct_orders_with_missing
450,WDID10322,Dana Ferguson,22,40,11,5,7.46,36.36
350,WDID10222,Daniel Hall,20,40,11,5,7.04,36.36
3,WDID09875,Stephen Randolph,18,64,11,6,6.52,36.36
12,WDID09884,James Winters,18,22,11,6,6.52,36.36
575,WDID10447,Patricia Vance,18,40,11,5,6.49,36.36
205,WDID10077,Brent Werner,19,15,11,5,6.33,36.36
207,WDID10079,Brittany Kelly,19,70,11,5,6.25,36.36
10,WDID09882,Angela Young,18,75,11,6,6.12,36.36
9,WDID09881,Tracey Watkins,18,26,11,6,6.06,36.36
549,WDID10421,Jesse Solomon,26,29,11,5,6.02,36.36


In [14]:
# Calculate Driver Risk Scores
driver_risk = get_driver_risk_scores(driver_features)

# Risk category distribution - HORIZONTAL BAR CHART
risk_dist = driver_risk['risk_category'].value_counts()
# Sort by risk order (Critical to Low)
risk_order = PROJECT_THEME['risk_order']
risk_dist = risk_dist.reindex([cat for cat in risk_order if cat in risk_dist.index])

# Create horizontal bar chart with risk colors
fig = go.Figure()
fig.add_trace(go.Bar(
    x=risk_dist.values,
    y=risk_dist.index,
    orientation='h',
    marker=dict(color=[PROJECT_THEME['risk_colors'][cat] for cat in risk_dist.index]),
    text=[f"{val} ({val/risk_dist.sum()*100:.1f}%)" for val in risk_dist.values],
    textposition='outside',
    textfont=dict(size=11)
))

fig.update_layout(
    title='Driver Risk Category Distribution',
    xaxis_title='Number of Drivers',
    yaxis_title='Risk Category',
    height=400,
    showlegend=False
)
apply_project_theme(fig)
fig.show()

print('Risk Categories:')
for cat in ['Low', 'Medium', 'High', 'Critical']:
    count = (driver_risk['risk_category'] == cat).sum()
    print(f"  {cat}: {count} drivers ({count/len(driver_risk)*100:.1f}%)")

Risk Categories:
  Low: 822 drivers (65.9%)
  Medium: 6 drivers (0.5%)
  High: 273 drivers (21.9%)
  Critical: 146 drivers (11.7%)


In [15]:
# Top High/Critical Risk Drivers
high_risk = driver_risk[driver_risk['risk_category'].isin(['High', 'Critical'])].sort_values('risk_score', ascending=False)

print(f"HIGH/CRITICAL RISK DRIVERS: {len(high_risk)}")
print('=' * 60)
display(high_risk[[
    'driver_id', 'driver_name', 'age', 'total_orders',
    'missing_rate', 'risk_score', 'risk_category'
]].head(10))

HIGH/CRITICAL RISK DRIVERS: 419


,driver_id,driver_name,age,total_orders,missing_rate,risk_score,risk_category
450,WDID10322,Dana Ferguson,22,11,7.46,95.83,Critical
3,WDID09875,Stephen Randolph,18,11,6.52,94.96,Critical
12,WDID09884,James Winters,18,11,6.52,94.96,Critical
350,WDID10222,Daniel Hall,20,11,7.04,93.58,Critical
10,WDID09882,Angela Young,18,11,6.12,92.82,Critical
9,WDID09881,Tracey Watkins,18,11,6.06,92.48,Critical
1,WDID09873,Pamela Moore,18,11,6.00,92.16,Critical
7,WDID09879,Shane Palmer,18,11,5.94,91.84,Critical
575,WDID10447,Patricia Vance,18,11,6.49,90.64,Critical
4,WDID09876,Jordan Daniel,18,11,5.71,90.63,Critical


---
## 3. Customer Analysis

Analyzing customer demographics and order behavior to identify potential fraud patterns.

In [16]:
# Create customer features
customer_features = create_customer_features(customers, orders)

print(f'Customer features created: {len(customer_features)} customers')
print(f'\nFeatures: {list(customer_features.columns)}')
customer_features.head()

Customer features created: 1239 customers

Features: ['customer_id', 'customer_name', 'customer_age', 'total_orders', 'total_spent', 'avg_order_value', 'total_items_delivered', 'total_items_missing', 'total_items_ordered', 'missing_rate', 'orders_with_missing', 'pct_orders_with_missing', 'age_group', 'customer_segment']


,customer_id,customer_name,customer_age,total_orders,total_spent,avg_order_value,total_items_delivered,total_items_missing,total_items_ordered,missing_rate,orders_with_missing,pct_orders_with_missing,age_group,customer_segment
0,WCID5000,Dustin Spencer,38,7,2092.20,298.89,82,0,82,0.00,0,0.00,36-45,Medium Value
1,WCID5001,Clayton Huff,66,9,2595.46,288.38,53,3,56,5.36,3,33.33,65+,High Value
2,WCID5002,Angelica Wilson,72,6,2196.25,366.04,76,3,79,3.80,2,33.33,65+,High Value
3,WCID5003,Lisa Marquez,20,11,3163.13,287.56,112,2,114,1.75,2,18.18,18-25,VIP
4,WCID5004,James Martinez,35,15,4520.94,301.40,155,3,158,1.90,3,20.00,26-35,VIP


In [17]:
# Customer Overview Statistics
active_customers = customer_features[customer_features['total_orders'] > 0]

print('=' * 60)
print('CUSTOMER OVERVIEW')
print('=' * 60)
print(f"Total Customers: {len(customer_features):,}")
print(f"Active Customers (with orders): {len(active_customers):,}")
print(f"Average Age: {customer_features['customer_age'].mean():.1f} years")
print(f"\nOrder Stats:")
print(f"  Average Orders per Customer: {active_customers['total_orders'].mean():.1f}")
print(f"  Average Spending: ${active_customers['total_spent'].mean():,.2f}")
print(f"  Average Missing Rate: {active_customers['missing_rate'].mean():.2f}%")
print(f"  Customers with Missing Issues: {(active_customers['orders_with_missing'] > 0).sum():,}")

CUSTOMER OVERVIEW
Total Customers: 1,239
Active Customers (with orders): 1,239
Average Age: 54.4 years

Order Stats:
  Average Orders per Customer: 8.1
  Average Spending: $2,286.54
  Average Missing Rate: 1.71%
  Customers with Missing Issues: 881


In [18]:
# Customer Age Distribution
fig = px.histogram(
    customer_features, 
    x='customer_age',
    nbins=25,
    title='Customer Age Distribution',
    labels={'customer_age': 'Age (years)', 'count': 'Number of Customers'},
    color_discrete_sequence=[PROJECT_THEME['walmart_blue']]
)
fig.add_vline(x=customer_features['customer_age'].mean(), line_dash='dash', line_color='red',
              annotation_text=f"Mean: {customer_features['customer_age'].mean():.1f}")
fig.update_layout(height=400)
apply_project_theme(fig)
fig.show()

In [19]:
# Customer Spending Distribution
fig = px.histogram(
    active_customers, 
    x='total_spent',
    nbins=40,
    title='Customer Total Spending Distribution',
    labels={'total_spent': 'Total Spent ($)', 'count': 'Number of Customers'},
    color_discrete_sequence=[PROJECT_THEME['walmart_blue']]
)
fig.update_layout(height=400)
apply_project_theme(fig)
fig.show()

In [20]:
# Customer Missing Rate Distribution
fig = px.histogram(
    active_customers, 
    x='missing_rate',
    nbins=30,
    title='Distribution of Missing Rate by Customer',
    labels={'missing_rate': 'Missing Rate (%)', 'count': 'Number of Customers'},
    color_discrete_sequence=[PROJECT_THEME['walmart_blue']]
)
avg_rate = active_customers['missing_rate'].mean()
fig.add_vline(x=avg_rate, line_dash='dash', line_color='red',
              annotation_text=f"Mean: {avg_rate:.2f}%")
fig.update_layout(height=400)
apply_project_theme(fig)
fig.show()

print(f"\nCustomers with missing rate > 30%: {(active_customers['missing_rate'] > 30).sum()}")
print(f"Customers with missing rate > 50%: {(active_customers['missing_rate'] > 50).sum()}")
print(f"Customers with 100% missing rate: {(active_customers['missing_rate'] == 100).sum()}")


Customers with missing rate > 30%: 0
Customers with missing rate > 50%: 0
Customers with 100% missing rate: 0


In [21]:
# Missing Rate vs Spending
fig = px.scatter(
    active_customers,
    x='total_spent',
    y='missing_rate',
    color='customer_segment',
    size='total_orders',
    hover_data=['customer_name', 'orders_with_missing', 'total_items_missing'],
    title='Missing Rate vs Total Spending',
    labels={'total_spent': 'Total Spent ($)', 'missing_rate': 'Missing Rate (%)', 'customer_segment': 'Segment'}
)
fig.update_layout(height=500)
apply_project_theme(fig)
fig.show()

In [22]:
# Missing Rate by Age Group
cust_age_analysis = active_customers.groupby('age_group', observed=True).agg({
    'missing_rate': 'mean',
    'customer_id': 'count',
    'total_spent': 'sum',
    'total_items_missing': 'sum'
}).reset_index()
cust_age_analysis.columns = ['age_group', 'avg_missing_rate', 'customer_count', 'total_spent', 'total_missing']

fig = px.bar(
    cust_age_analysis,
    x='age_group',
    y='avg_missing_rate',
    title='Average Missing Rate by Customer Age Group',
    labels={'avg_missing_rate': 'Average Missing Rate (%)', 'age_group': 'Age Group'},
    color='avg_missing_rate',
    color_continuous_scale='Reds',
    text='avg_missing_rate'
)
fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(height=400)
apply_project_theme(fig)
fig.show()

In [23]:
# Missing Rate by Customer Segment
segment_analysis = active_customers.groupby('customer_segment', observed=True).agg({
    'missing_rate': 'mean',
    'customer_id': 'count',
    'total_spent': 'sum',
    'orders_with_missing': 'sum'
}).reset_index()
segment_analysis.columns = ['segment', 'avg_missing_rate', 'customer_count', 'total_revenue', 'total_issues']

fig = px.bar(
    segment_analysis,
    x='segment',
    y='avg_missing_rate',
    title='Average Missing Rate by Customer Segment',
    labels={'avg_missing_rate': 'Average Missing Rate (%)', 'segment': 'Customer Segment'},
    color='avg_missing_rate',
    color_continuous_scale='Purples',
    text='avg_missing_rate'
)
fig.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig.update_layout(height=400)
apply_project_theme(fig)
fig.show()

print('\nSegment Details:')
display(segment_analysis)


Segment Details:


,segment,avg_missing_rate,customer_count,total_revenue,total_issues
0,Low Value,1.77,310,367369.26,232
1,Medium Value,1.64,310,585667.29,321
2,High Value,1.69,309,767665.12,406
3,VIP,1.75,310,1112320.71,543


In [24]:
# Identify Suspicious Customers
suspicious_customers = get_suspicious_customers(customer_features, missing_rate_threshold=25, min_orders=2)

print('=' * 60)
print('SUSPICIOUS CUSTOMERS')
print('=' * 60)
print(f"Total suspicious customers: {len(suspicious_customers)}")
print(f"Percentage of active customers: {len(suspicious_customers)/len(active_customers)*100:.1f}%")
print()

# Show top 15 suspicious customers
display(suspicious_customers[[
    'customer_id', 'customer_name', 'customer_age', 'customer_segment',
    'total_orders', 'total_spent', 'missing_rate', 'orders_with_missing'
]].head(15))

SUSPICIOUS CUSTOMERS
Total suspicious customers: 46
Percentage of active customers: 3.7%



,customer_id,customer_name,customer_age,customer_segment,total_orders,total_spent,missing_rate,orders_with_missing
743,WCID5743,Justin Williams,78,Low Value,4,917.03,13.64,2
1043,WCID6043,Joshua Thompson,79,Low Value,3,702.76,11.54,2
762,WCID5762,Matthew Osborne,74,High Value,7,2384.49,10.00,3
711,WCID5711,Matthew Barker,46,Low Value,3,970.23,9.52,2
18,WCID5018,Mark Mcmahon,78,Low Value,5,1201.68,9.09,2
1136,WCID6136,Robert Skinner,89,Low Value,5,1109.19,9.09,2
698,WCID5698,Crystal Smith,44,Low Value,4,777.93,8.89,3
905,WCID5905,Julie Reese,80,High Value,7,2203.24,7.94,3
594,WCID5594,Robert Odonnell,45,Medium Value,7,2075.43,7.81,3
630,WCID5630,Stephanie Harris,67,High Value,6,2837.59,7.69,2


In [25]:
# Calculate Customer Risk Scores
customer_risk = get_customer_risk_scores(customer_features)

# Risk category distribution - HORIZONTAL BAR CHART
cust_risk_dist = customer_risk['risk_category'].value_counts()
# Sort by risk order (Critical to Low)
risk_order = PROJECT_THEME['risk_order']
cust_risk_dist = cust_risk_dist.reindex([cat for cat in risk_order if cat in cust_risk_dist.index])

# Create horizontal bar chart with risk colors
fig = go.Figure()
fig.add_trace(go.Bar(
    x=cust_risk_dist.values,
    y=cust_risk_dist.index,
    orientation='h',
    marker=dict(color=[PROJECT_THEME['risk_colors'][cat] for cat in cust_risk_dist.index]),
    text=[f"{val} ({val/cust_risk_dist.sum()*100:.1f}%)" for val in cust_risk_dist.values],
    textposition='outside',
    textfont=dict(size=11)
))

fig.update_layout(
    title='Customer Risk Category Distribution',
    xaxis_title='Number of Customers',
    yaxis_title='Risk Category',
    height=400,
    showlegend=False
)
apply_project_theme(fig)
fig.show()

print('Risk Categories:')
for cat in ['Low', 'Medium', 'High', 'Critical']:
    count = (customer_risk['risk_category'] == cat).sum()
    print(f"  {cat}: {count} customers ({count/len(customer_risk)*100:.1f}%)")

Risk Categories:
  Low: 995 customers (80.3%)
  Medium: 228 customers (18.4%)
  High: 15 customers (1.2%)
  Critical: 1 customers (0.1%)


In [26]:
# Top High/Critical Risk Customers
high_risk_cust = customer_risk[customer_risk['risk_category'].isin(['High', 'Critical'])].sort_values('risk_score', ascending=False)

print(f"HIGH/CRITICAL RISK CUSTOMERS: {len(high_risk_cust)}")
print('=' * 60)
display(high_risk_cust[[
    'customer_id', 'customer_name', 'customer_age', 'customer_segment',
    'total_orders', 'missing_rate', 'risk_score', 'risk_category'
]].head(10))

HIGH/CRITICAL RISK CUSTOMERS: 16


,customer_id,customer_name,customer_age,customer_segment,total_orders,missing_rate,risk_score,risk_category
903,WCID5903,Sharon Allen,34,Low Value,1,12.50,75.24,Critical
743,WCID5743,Justin Williams,78,Low Value,4,13.64,64.64,High
1043,WCID6043,Joshua Thompson,79,Low Value,3,11.54,64.32,High
698,WCID5698,Crystal Smith,44,Low Value,4,8.89,63.04,High
685,WCID5685,Shelley Gould,85,High Value,12,4.93,59.88,High
711,WCID5711,Matthew Barker,46,Low Value,3,9.52,58.41,High
820,WCID5820,Edward Henry,73,VIP,15,5.26,56.77,High
923,WCID5923,Kenneth Brown,19,Low Value,7,7.14,55.24,High
762,WCID5762,Matthew Osborne,74,High Value,7,10.00,55.05,High
672,WCID5672,Dana Gutierrez,19,Medium Value,7,6.67,53.84,High


---
## 4. Driver-Customer Interaction Analysis

Detecting potential collusion patterns between drivers and customers.

In [27]:
# Create driver-customer interaction matrix
interactions, suspicious_pairs = create_driver_customer_matrix(orders)

print('=' * 60)
print('DRIVER-CUSTOMER INTERACTIONS')
print('=' * 60)
print(f"Total unique driver-customer pairs: {len(interactions):,}")
print(f"Average interactions per pair: {interactions['interactions'].mean():.2f}")
print(f"Max interactions for a pair: {interactions['interactions'].max()}")
print(f"\nSuspicious pairs (high interaction + high missing rate): {len(suspicious_pairs)}")

DRIVER-CUSTOMER INTERACTIONS
Total unique driver-customer pairs: 9,967
Average interactions per pair: 1.00
Max interactions for a pair: 2

Suspicious pairs (high interaction + high missing rate): 14


In [28]:
# Interaction Distribution
fig = px.histogram(
    interactions,
    x='interactions',
    nbins=20,
    title='Distribution of Driver-Customer Interactions',
    labels={'interactions': 'Number of Interactions', 'count': 'Number of Pairs'},
    color_discrete_sequence=[PROJECT_THEME['walmart_blue']]
)
fig.update_layout(height=400)
apply_project_theme(fig)
fig.show()

# Stats
print(f"Pairs with only 1 interaction: {(interactions['interactions'] == 1).sum()} ({(interactions['interactions'] == 1).sum()/len(interactions)*100:.1f}%)")
print(f"Pairs with 2+ interactions: {(interactions['interactions'] >= 2).sum()}")
print(f"Pairs with 3+ interactions: {(interactions['interactions'] >= 3).sum()}")

Pairs with only 1 interaction: 9934 (99.7%)
Pairs with 2+ interactions: 33
Pairs with 3+ interactions: 0


In [29]:
# Repeated Interactions Analysis
repeated = interactions[interactions['interactions'] >= 2].copy()

fig = px.scatter(
    repeated,
    x='interactions',
    y='missing_rate',
    size='items_missing',
    color='missing_rate',
    color_continuous_scale='Reds',
    hover_data=['driver_id', 'customer_id', 'items_missing'],
    title='Missing Rate in Repeated Driver-Customer Interactions',
    labels={'interactions': 'Number of Interactions', 'missing_rate': 'Missing Rate (%)'}
)
fig.update_layout(height=500)
apply_project_theme(fig)
fig.show()

In [30]:
# Suspicious Pairs (Potential Collusion)
print('=' * 70)
print('SUSPICIOUS DRIVER-CUSTOMER PAIRS (POTENTIAL COLLUSION)')
print('=' * 70)
print(f"Criteria: Above average interactions AND missing rate > 1.5x average")
print(f"Found: {len(suspicious_pairs)} suspicious pairs")
print()

if len(suspicious_pairs) > 0:
    # Enrich with names
    susp_enriched = suspicious_pairs.merge(
        drivers[['driver_id', 'driver_name']], on='driver_id', how='left'
    ).merge(
        customers[['customer_id', 'customer_name']], on='customer_id', how='left'
    )
    
    display(susp_enriched[[
        'driver_id', 'driver_name', 'customer_id', 'customer_name',
        'interactions', 'items_missing', 'missing_rate'
    ]].head(15))

SUSPICIOUS DRIVER-CUSTOMER PAIRS (POTENTIAL COLLUSION)
Criteria: Above average interactions AND missing rate > 1.5x average
Found: 14 suspicious pairs



,driver_id,driver_name,customer_id,customer_name,interactions,items_missing,missing_rate
0,WDID10225,Mary Miller,WCID6128,Robert Wilson,2,2,14.29
1,WDID10626,Michael Wise,WCID6213,Ryan Warren,2,1,12.50
2,WDID10096,Kelly Lopez,WCID5985,James Berry,2,2,9.52
3,WDID10671,Justin Villa,WCID5260,Danielle Boyer,2,2,9.09
4,WDID10569,Alexandra Jones,WCID5339,Jerry Mullen,2,1,7.69
5,WDID10546,Crystal Bryant,WCID5071,Anthony Rodriguez,2,1,6.67
6,WDID10613,Jennifer Miller,WCID6082,Victor Davis,2,1,6.25
7,WDID10597,Stephen Krueger,WCID5637,Nancy Torres,2,1,5.88
8,WDID10515,Kelly Anderson,WCID5585,Robert Vasquez,2,1,5.26
9,WDID10547,Laura Coleman,WCID6178,Susan Reilly,2,1,5.00


In [31]:
# Heatmap of Top Interacting Pairs
top_drivers = interactions.groupby('driver_id')['interactions'].sum().nlargest(15).index
top_customers = interactions.groupby('customer_id')['interactions'].sum().nlargest(15).index

matrix_data = interactions[
    interactions['driver_id'].isin(top_drivers) & 
    interactions['customer_id'].isin(top_customers)
]

# Create pivot for heatmap
heatmap_pivot = matrix_data.pivot_table(
    values='missing_rate', 
    index='driver_id', 
    columns='customer_id',
    fill_value=0
)

# FIXED: Use 'Blues' sequential colormap instead of divergent
fig = px.imshow(
    heatmap_pivot,
    title='Missing Rate Heatmap: Top Interacting Drivers vs Customers',
    labels={'color': 'Missing Rate (%)'},
    color_continuous_scale='Blues',
    aspect='auto'
)
fig.update_layout(height=500)
apply_project_theme(fig)
fig.show()

---
## 5. Key Findings and Summary

In [33]:
# Summary Statistics
print('=' * 70)
print('KEY FINDINGS - DRIVERS & CUSTOMERS EDA')
print('=' * 70)

# Driver stats
print(f"\n1. DRIVER ANALYSIS")
print(f"   - Total Drivers: {len(drivers):,}")
print(f"   - Active Drivers: {len(active_drivers):,}")
print(f"   - Average Missing Rate: {active_drivers['missing_rate'].mean():.2f}%")
print(f"   - Suspicious Drivers (>15% missing): {len(suspicious_drivers)}")
print(f"   - High/Critical Risk Drivers: {len(high_risk)}")

# Highest risk age group for drivers
highest_risk_age = age_analysis.loc[age_analysis['avg_missing_rate'].idxmax()]
print(f"   - Highest Risk Age Group: {highest_risk_age['age_group']} ({highest_risk_age['avg_missing_rate']:.2f}%)")

# Customer stats
print(f"\n2. CUSTOMER ANALYSIS")
print(f"   - Total Customers: {len(customers):,}")
print(f"   - Active Customers: {len(active_customers):,}")
print(f"   - Average Missing Rate: {active_customers['missing_rate'].mean():.2f}%")
print(f"   - Suspicious Customers (>25% missing): {len(suspicious_customers)}")
print(f"   - High/Critical Risk Customers: {len(high_risk_cust)}")

# Highest risk segment
highest_risk_seg = segment_analysis.loc[segment_analysis['avg_missing_rate'].idxmax()]
print(f"   - Highest Risk Segment: {highest_risk_seg['segment']} ({highest_risk_seg['avg_missing_rate']:.2f}%)")

# Collusion indicators
print(f"\n3. COLLUSION INDICATORS")
print(f"   - Total Driver-Customer Pairs: {len(interactions):,}")
print(f"   - Pairs with Repeated Interactions: {(interactions['interactions'] >= 2).sum()}")
print(f"   - Suspicious Pairs Detected: {len(suspicious_pairs)}")

# Estimated losses
total_missing_items = orders['items_missing'].sum()
avg_item_value = 15  # Estimated average item value
print(f"\n4. ESTIMATED IMPACT")
print(f"   - Total Items Missing: {total_missing_items:,}")
print(f"   - Estimated Loss (at ${avg_item_value}/item): ${total_missing_items * avg_item_value:,.2f}")

KEY FINDINGS - DRIVERS & CUSTOMERS EDA

1. DRIVER ANALYSIS
   - Total Drivers: 1,247
   - Active Drivers: 1,247
   - Average Missing Rate: 1.22%
   - Suspicious Drivers (>15% missing): 54
   - High/Critical Risk Drivers: 419
   - Highest Risk Age Group: 55+ (1.42%)

2. CUSTOMER ANALYSIS
   - Total Customers: 1,239
   - Active Customers: 1,239
   - Average Missing Rate: 1.71%
   - Suspicious Customers (>25% missing): 46
   - High/Critical Risk Customers: 16
   - Highest Risk Segment: Low Value (1.77%)

3. COLLUSION INDICATORS
   - Total Driver-Customer Pairs: 9,967
   - Pairs with Repeated Interactions: 33
   - Suspicious Pairs Detected: 14

4. ESTIMATED IMPACT
   - Total Items Missing: 1,657
   - Estimated Loss (at $15/item): $24,855.00


---
## Summary

### Key Insights from Drivers & Customers EDA:

#### Drivers:
1. **Suspicious drivers identified** - A subset of drivers shows significantly higher missing rates than average
2. **Experience matters** - Analysis shows correlation between experience level and missing rate
3. **Age group patterns** - Certain age groups show higher fraud indicators
4. **Risk scoring** - Risk categories help prioritize investigation

#### Customers:
1. **Repeat offenders** - Some customers consistently report missing items
2. **Segment analysis** - Missing rates vary by customer value segment
3. **High-value targets** - Some customers may be gaming the system

#### Collusion:
1. **Suspicious pairs detected** - Driver-customer pairs with high interaction and high missing rates
2. **Pattern detection** - Repeated interactions with consistent missing items suggest possible collusion

### Next Steps:
- Deep dive into fraud patterns (Notebook 03)
- Build ML models for fraud detection (Notebook 04)
- Create monitoring dashboard